# Diagnostic Unifié ETCCDI — Protocole complet validé par l'équipe de 5 experts

## Objectif

1. **Trancher** la discrepance F1@p99 = 0.539 (diag) vs 0.175 (phase6_final) via grille factorielle.
2. **Réévaluer** le Phase 6 dual-path en mode ETCCDI per-pixel climatology (publication-ready).
3. **Croiser** avec CSI/SEDI/FSS pour robustesse.

## Les 4 étapes (un seul run)

| Étape | Description | Coût |
|-------|-------------|------|
| **0** | Construire le vrai `land_mask_NZ` depuis HR brut NetCDF (AVANT le pipeline `fillna`) | ~5 s |
| **1** | Climatologie p95/p99 per-pixel sur train 1980-2009 wet days (≥ 1 mm/jour) | ~30 s |
| **2** | **Sampling unique** : N=16 batches × K=64 samples sur Phase 6 dual-path live | ~25 min |
| **3a** | **Grille factorielle 2×2×2** post-hoc : (NaN-filter, NaN-zero) × (N=4, N=16) × (K=32, K=64) | gratuit (post-hoc) |
| **3b** | F1 ETCCDI per-pixel avec `clim_p99` + CSI + SEDI + FSS | gratuit |
| **4** | Reload JSONs existants (noncausal v4, V5, seed_42 v1/v2) + synthèse honnête | ~5 s |

## Sortie

`oracle_9node/seed_42/unified_etccdi_results.json` contenant :
- Tableau factoriel complet (8 cellules × {F1, threshold, TP/FP/FN, bootstrap CI})
- Métriques ETCCDI Phase 6 dual-path (per-pixel clim threshold) + croisement
- Comparatif vs JSON existants (avec caveat convention)
- Verdict synthétique sur la discrepance

## Limitations honnêtes

- Étape 3b en mode ETCCDI per-pixel n'est appliquée qu'au Phase 6 dual-path. Pour les autres modèles (noncausal v4, V5, etc.) on **lit les JSON existants** (convention non-ETCCDI). Pour comparer publication-ready, il faudra **ré-évaluer ces modèles** dans un run séparé.

In [ ]:
# === Cell 1 : Bootstrap Colab ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric, cftime, h5netcdf, xbatcher, diffusers
    from omegaconf import OmegaConf
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ], check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab')

import torch, numpy as np, json, time
import xarray as xr
from omegaconf import OmegaConf
# Expert IA-B3 fix : reproducibility on GPU
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f'cwd={os.getcwd()}  torch={torch.__version__}  cuda={torch.cuda.is_available()}')

In [ ]:
# === Cell 2 : Constants ===
DRIVE_ROOT = Path('/content/drive/MyDrive/climate_data')
ORACLE_9N  = DRIVE_ROOT / 'oracle_9node' / 'seed_42'
CKPT_DUALPATH = ORACLE_9N / 'epoch_best_dualpath.pth'   # Stage 1 dual-path frozen
CKPT_STAGE2   = ORACLE_9N / 'epoch_last.pth'            # Stage 2 V5 (live weights)
SIGMA_DATA_NEW = 0.193

DATA_ROOT = DRIVE_ROOT / 'data'
HR_RAW_PATH = DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc'

OUT_DIR = ORACLE_9N / 'diagnostic_unified_etccdi'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_JSON = OUT_DIR / 'unified_etccdi_results.json'
LAND_MASK_PATH = OUT_DIR / 'land_mask_nz.npy'
CLIM_PATH      = OUT_DIR / 'clim_train_p95_p99_per_pixel.npz'

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# BS30 full protocol
N_BATCHES_FULL = 16
K_SAMPLES_FULL = 64
N_STEPS_DIFF   = 18

TRAIN_START = '1980-01-01'
TRAIN_END   = '2009-12-31'

WET_DAY_THRESHOLD_MM = 1.0   # ETCCDI standard
PRECIPITATION_DELTA = 0.01   # pipeline applies log1p(x + 0.01) -- inverse must subtract

print(f'DEVICE = {DEVICE}')
print(f'CKPT_STAGE2 = {CKPT_STAGE2}  exists={CKPT_STAGE2.exists()}')
print(f'CKPT_DUALPATH = {CKPT_DUALPATH}  exists={CKPT_DUALPATH.exists()}')
print(f'HR_RAW_PATH = {HR_RAW_PATH}  exists={HR_RAW_PATH.exists()}')
print(f'OUT_JSON = {OUT_JSON}')

In [ ]:
# === Cell 3 : ÉTAPE 0 — Land mask depuis HR brut (AVANT pipeline fillna) ===
# Le pipeline applique nan_fill_strategy="mean" qui remplit les NaN océan AVANT
# l'éval, donc on perd la séparation terre/mer. On reload le NetCDF brut.

_t0 = time.time()
ds_hr = xr.open_dataset(str(HR_RAW_PATH), engine='h5netcdf')
print(f'[Étape 0] HR raw dataset variables : {list(ds_hr.data_vars)}')
_pr_var = 'pr' if 'pr' in ds_hr.data_vars else list(ds_hr.data_vars)[0]
_hr = ds_hr[_pr_var]
print(f'[Étape 0] shape brute = {tuple(_hr.shape)}  dtype = {_hr.dtype}')

# Land mask = pixels where HR is NOT NaN on the first time step
# (l'océan a NaN persistent ; la terre a des valeurs définies même sec).
_hr_t0 = _hr.isel({_hr.dims[0]: 0}).values
land_mask = ~np.isnan(_hr_t0)
n_land = int(land_mask.sum())
n_total = int(land_mask.size)
print(f'[Étape 0] land pixels = {n_land} / {n_total}  ({100*n_land/n_total:.1f}%)')
print(f'[Étape 0] ocean pixels = {n_total - n_land}  ({100*(n_total - n_land)/n_total:.1f}%)')

np.save(LAND_MASK_PATH, land_mask)
print(f'[Étape 0] saved : {LAND_MASK_PATH}')
print(f'[Étape 0] done in {time.time()-_t0:.1f}s')

In [ ]:
# === Cell 4 : ÉTAPE 1 — Climatologie p95/p99 per-pixel sur train wet days ===
# Standard ETCCDI : p99 of wet days (>= 1 mm/day) for each pixel separately,
# computed on the training period only (avoids leakage).

_t0 = time.time()
# Slice train period
_time_var = _hr.dims[0]
_hr_train = _hr.sel({_time_var: slice(TRAIN_START, TRAIN_END)})
print(f'[Étape 1] train slice = {_hr_train.shape[0]} time steps  ({TRAIN_START} -> {TRAIN_END})')

_hr_train_np = _hr_train.values.astype(np.float32)   # [T, H, W] in mm/day
print(f'[Étape 1] train tensor shape = {_hr_train_np.shape}  size = {_hr_train_np.nbytes/1e9:.2f} GB')

# Per-pixel p95 / p99 of wet days only.
H, W = _hr_train_np.shape[1], _hr_train_np.shape[2]
clim_p95 = np.full((H, W), np.nan, dtype=np.float32)
clim_p99 = np.full((H, W), np.nan, dtype=np.float32)
n_wet_per_pixel = np.zeros((H, W), dtype=np.int32)

for i in range(H):
    for j in range(W):
        if not land_mask[i, j]:
            continue
        px = _hr_train_np[:, i, j]
        px_finite = px[np.isfinite(px)]
        wet = px_finite[px_finite >= WET_DAY_THRESHOLD_MM]
        n_wet_per_pixel[i, j] = wet.size
        if wet.size >= 30:    # need enough wet days for a meaningful quantile
            clim_p95[i, j] = float(np.quantile(wet, 0.95))
            clim_p99[i, j] = float(np.quantile(wet, 0.99))

_n_land_with_clim = int(np.isfinite(clim_p99).sum())
print(f'[Étape 1] land pixels with valid p99 = {_n_land_with_clim} / {n_land}')
print(f'[Étape 1] mean wet days per land pixel = {n_wet_per_pixel[land_mask].mean():.1f}')
print(f'[Étape 1] clim_p95 range = [{np.nanmin(clim_p95):.2f}, {np.nanmax(clim_p95):.2f}] mm/day')
print(f'[Étape 1] clim_p99 range = [{np.nanmin(clim_p99):.2f}, {np.nanmax(clim_p99):.2f}] mm/day')
print(f'[Étape 1] clim_p99 mean = {np.nanmean(clim_p99):.2f} mm/day')

np.savez(CLIM_PATH, clim_p95=clim_p95, clim_p99=clim_p99,
         n_wet_per_pixel=n_wet_per_pixel, land_mask=land_mask,
         wet_threshold_mm=WET_DAY_THRESHOLD_MM,
         train_start=TRAIN_START, train_end=TRAIN_END)
print(f'[Étape 1] saved : {CLIM_PATH}')
print(f'[Étape 1] done in {time.time()-_t0:.1f}s')

del _hr_train_np
ds_hr.close()

In [ ]:
# === Cell 5 : Pipeline + Stage 1 dual-path FROZEN ===
from torch.utils.data import DataLoader as _DataLoader, IterableDataset
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from st_cdgm.models.dual_path_stage1 import DualPathPredictor
from st_cdgm.training.stage1_paths import batch_lr_grid_last
from st_cdgm.models.intelligible_encoder import IntelligibleVariableEncoder, IntelligibleVariableConfig
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES

CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)
CONFIG.training.batch_size  = 1
CONFIG.training.use_amp     = True
CONFIG.training.num_workers = 0
ts_cfg = CONFIG.two_stage
ts_cfg.stage1['lambda_dag_prior'] = PATHCPLUS_HYPERPARAM_OVERRIDES['lambda_dag_prior']
ts_cfg.stage1['g_phys_alpha']     = PATHCPLUS_HYPERPARAM_OVERRIDES['g_phys_alpha']
OmegaConf.set_struct(CONFIG, False)
for _m in [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]:
    if _m['name'] not in {mm.name for mm in CONFIG.encoder.metapaths}:
        CONFIG.encoder.metapaths.append(OmegaConf.create(_m))

K9_DATES = {'train': [TRAIN_START, TRAIN_END], 'val': ['2010-01-01', '2011-12-31'],
            'test': ['2012-01-01', '2013-12-31'], 'holdout': ['2014-01-01', '2014-12-31']}
LR_PATH = str(DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc')
_static = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'
_mean   = DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc'
_std    = DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc'

SEQ_LEN = int(CONFIG.data.seq_len)
pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH, hr_path=str(HR_RAW_PATH),
    static_path=str(_static) if _static.exists() else None,
    seq_len=SEQ_LEN, baseline_strategy=str(CONFIG.data.baseline_strategy),
    baseline_factor=int(CONFIG.data.baseline_factor),
    normalize=bool(CONFIG.data.normalize),
    nan_fill_strategy=str(CONFIG.data.nan_fill_strategy),
    precipitation_delta=float(CONFIG.data.precipitation_delta),
    lr_variables=list(CONFIG.data.lr_variables), hr_variables=list(CONFIG.data.hr_variables),
    static_variables=list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else [],
    means_path=str(_mean) if _mean.exists() else None,
    stds_path=str(_std) if _std.exists() else None,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)
test_dataset = pipeline.build_sequence_dataset(split='test', seq_len=SEQ_LEN,
                                                stride=int(CONFIG.data.stride), as_torch=True)
val_dataset  = pipeline.build_sequence_dataset(split='val', seq_len=SEQ_LEN,
                                                stride=int(CONFIG.data.stride), as_torch=True)

lr_shape = tuple(CONFIG.graph.lr_shape); hr_shape = tuple(CONFIG.graph.hr_shape)
builder = HeteroGraphBuilder(lr_shape=lr_shape, hr_shape=hr_shape,
                              static_dataset=pipeline.get_static_dataset(),
                              include_mid_layer=CONFIG.graph.include_mid_layer,
                              extended_9node=True)
H_HR, W_HR = int(hr_shape[0]), int(hr_shape[1])

# ---- Helper convert_sample_to_batch (mirror phase6) ----
_LR_VARS = list(CONFIG.data.lr_variables)
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850','q_500','q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850','w_500','w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850','500','250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, [_VI[f'q_{lev}']]]; u = lr0[:, [_VI[f'u_{lev}']]]; v = lr0[:, [_VI[f'v_{lev}']]]
        term = q * torch.sqrt(u*u + v*v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None: acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def _ensure_2d(t): return t.unsqueeze(-1) if t.dim() == 1 else t

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample['lr']; seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    lr0 = lr_nodes_steps[0]
    _ivt = _compute_ivt_nodes(lr0)
    dyn = {}
    for nt in builder.dynamic_node_types:
        if nt == 'Q850':   dyn[nt] = _ensure_2d(lr0[:, _Q_IDX] if _Q_IDX else lr0)
        elif nt == 'W500': dyn[nt] = _ensure_2d(lr0[:, _W_IDX] if _W_IDX else lr0)
        elif nt == 'IVT':  dyn[nt] = _ensure_2d(_ivt)
        else:              dyn[nt] = _ensure_2d(lr0)
    hetero = builder.prepare_step_data(dyn).to(device)
    return {'lr': lr_tensor, 'lr_grid': lr_seq, 'residual': sample['residual'],
            'baseline': sample.get('baseline'), 'hetero': hetero, 'time': sample.get('time')}

# ---- Load Stage 1 dual-path FROZEN ----
def _clean_sd(sd):
    if sd is None: return None
    if any('_orig_mod' in k for k in sd):
        sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
    return sd
def _strip_prefixes(sd):
    if sd is None: return None
    out = {}
    for k, v in sd.items():
        nk = k
        for p in ('_orig_mod.', 'module.'):
            if nk.startswith(p): nk = nk[len(p):]
        out[nk] = v
    return out
def _safe_load(module, ck, keys, label):
    for key in keys:
        sd = ck.get(key)
        if sd is not None:
            sd = _clean_sd(sd)
            try:
                m, u = module.load_state_dict(sd, strict=False)
                print(f'  [{label}] loaded from "{key}" missing={len(m)} unexpected={len(u)}')
                return True
            except Exception as e:
                print(f'  [{label}] FAILED "{key}" : {e}')
    return False

ck_s1 = torch.load(CKPT_DUALPATH, map_location=DEVICE, weights_only=False)
enc_sd = _clean_sd(ck_s1.get('encoder_state_dict', {}))
seen, order = {}, []
for k in enc_sd:
    if not k.startswith('metapath_convs.'): continue
    parts = k[len('metapath_convs.'):].split('__')
    if len(parts) < 4: continue
    name, src, rel, tgt = parts[0], parts[1], parts[2], parts[3].split('.')[0]
    if name not in seen: seen[name] = (src, rel, tgt); order.append(name)
cfgs = [IntelligibleVariableConfig(name=n, meta_path=(seen[n][0], seen[n][1], seen[n][2]), pool='mean') for n in order]
encoder = IntelligibleVariableEncoder(configs=cfgs, hidden_dim=int(CONFIG.encoder.hidden_dim),
                                       conditioning_dim=int(CONFIG.encoder.conditioning_dim)).to(DEVICE)
num_vars = len(cfgs)
_probe = next(iter(test_dataset))
C_LR = _probe['lr'].shape[1]
_lr_nodes = builder.lr_grid_to_nodes(_probe['lr'][0])
rcn_driver_dim = _lr_nodes.shape[-1]
rcn_cell = RCNCell(num_vars=num_vars, hidden_dim=int(CONFIG.rcn.hidden_dim),
                    driver_dim=rcn_driver_dim, reconstruction_dim=rcn_driver_dim,
                    dropout=float(CONFIG.rcn.dropout)).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval'))
rh_cfg = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(d_model=int(rh_cfg.d_model), hr_h=H_HR, hr_w=W_HR,
                                      intermediate_h=int(rh_cfg.intermediate_h),
                                      intermediate_w=int(rh_cfg.intermediate_w),
                                      n_heads=int(rh_cfg.n_heads),
                                      refine_channels=int(rh_cfg.refine_channels),
                                      output_channels=1).to(DEVICE)
dual_path = DualPathPredictor(in_channels=C_LR, base_ch=48, hr_h=H_HR, hr_w=W_HR,
                               gate_max_mean=0.40, path_b_kind='unet',
                               path_b_unet_channels=(32, 64, 128),
                               path_b_unet_lr_shape=(23, 26)).to(DEVICE)
_safe_load(encoder, ck_s1, ['encoder_state_dict'], 'encoder')
_safe_load(rcn_cell, ck_s1, ['rcn_cell_state_dict', 'rcn_state_dict'], 'rcn_cell')
_safe_load(regression_head, ck_s1, ['regression_head_state_dict', 'head_state_dict'], 'regression_head')
_safe_load(dual_path, ck_s1, ['dual_path_state_dict'], 'dual_path')
for m in [encoder, rcn_cell, regression_head, dual_path]:
    for p in m.parameters(): p.requires_grad_(False)
    m.eval()
print(f'[Cell 5] Stage 1 FROZEN. num_vars={num_vars}')

In [ ]:
# === Cell 6 : Stage 2 (live + EMA if present) ===
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from omegaconf import OmegaConf as _OC

ck_s2 = torch.load(CKPT_STAGE2, map_location=DEVICE, weights_only=False)
print(f'[Cell 6] ckpt epoch = {ck_s2.get("epoch", "?")}')
_has_live = ck_s2.get('diffusion_state_dict') is not None
_has_ema  = ck_s2.get('diffusion_ema_state_dict') is not None
print(f'[Cell 6] live ? {_has_live}  EMA ? {_has_ema}')

UNET_KW = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
for _k in ('down_block_types', 'up_block_types'):
    if _k in UNET_KW and isinstance(UNET_KW[_k], list):
        UNET_KW[_k] = tuple(UNET_KW[_k])
UNET_KW['projection_class_embeddings_input_dim'] = num_vars * int(CONFIG.diffusion.conditioning_dim)
edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get('edm', {}))
_probe = next(iter(val_dataset))
hr_channels = int(_probe['residual'].shape[1])

def _build_decoder():
    d = CausalDiffusionDecoder(
        in_channels=hr_channels,
        conditioning_dim=CONFIG.diffusion.conditioning_dim,
        height=int(CONFIG.diffusion.height), width=int(CONFIG.diffusion.width),
        unet_kwargs=UNET_KW,
        scheduler_type=str(CONFIG.diffusion.scheduler_type),
        use_gradient_checkpointing=False,
        conv_padding_mode=str(CONFIG.diffusion.get('conv_padding_mode', 'zeros')),
        anti_checkerboard=bool(CONFIG.diffusion.get('anti_checkerboard', False)),
        edm_config=edm_cfg, causal_concat=True,
    ).to(DEVICE)
    d.edm_config.sigma_data = float(SIGMA_DATA_NEW)
    return d

diff_live = _build_decoder()
m, u = diff_live.load_state_dict(_strip_prefixes(ck_s2['diffusion_state_dict']), strict=False)
print(f'[Cell 6] LIVE loaded   missing={len(m)} unexpected={len(u)}')
for p in diff_live.parameters(): p.requires_grad_(False)
diff_live.eval()

if _has_ema:
    diff_ema = _build_decoder()
    m, u = diff_ema.load_state_dict(_strip_prefixes(ck_s2['diffusion_ema_state_dict']), strict=False)
    print(f'[Cell 6] EMA  loaded   missing={len(m)} unexpected={len(u)}')
    for p in diff_ema.parameters(): p.requires_grad_(False)
    diff_ema.eval()
else:
    diff_ema = None

In [ ]:
# === Cell 7 : Helpers (sampling, F1 4 conventions, CSI, SEDI, FSS) ===
@torch.no_grad()
def compute_mu_total_and_target(batch):
    lr_data = batch['lr'].to(DEVICE)
    h_init  = encoder.init_state(batch['hetero']).to(DEVICE)
    drivers = [lr_data[t] for t in range(lr_data.shape[0])]
    seq_out = rcn_runner.run(h_init, drivers, reconstruction_sources=None)
    mu_A = regression_head(seq_out.states[-1])
    if mu_A.dim() == 3: mu_A = mu_A.unsqueeze(0)
    lr_grid = batch_lr_grid_last(batch, builder=builder, device=DEVICE)
    lr_safe = torch.nan_to_num(lr_grid, nan=0.0)
    mu_total, mu_B, gate = dual_path(lr_safe, mu_A)
    mu_total = torch.nan_to_num(mu_total, nan=0.0)
    bl = batch['baseline'][-1].to(DEVICE)
    if bl.dim() == mu_total.dim() - 1: bl = bl.unsqueeze(0)
    bl = torch.nan_to_num(bl, nan=0.0)
    tgt = batch['residual'][-1].to(DEVICE)
    if tgt.dim() == 3: tgt = tgt.unsqueeze(0)
    return mu_total, bl, tgt

def _sample_once(decoder, mu_HR, baseline_log):
    out = decoder.sample(
        conditioning=None, num_steps=N_STEPS_DIFF,
        scheduler_type='edm_karras', cfg_scale=0.0,
        apply_constraints=False,
        mu_HR=mu_HR, baseline_log=baseline_log,
    )
    return out.residual if hasattr(out, 'residual') else out

# ----- F1 with 4 conventions -----
def f1_filter(pred_flat, target_flat, valid_flat, percentile, strict=True):
    """Convention A : filter NaN, then quantile on valid-only."""
    pv = pred_flat[valid_flat]; tv = target_flat[valid_flat]
    if tv.numel() < 100: return float('nan'), float('nan')
    thr = float(torch.quantile(tv, percentile / 100.0))
    cmp = (lambda x: x > thr) if strict else (lambda x: x >= thr)
    P = cmp(pv).float(); Y = cmp(tv).float()
    tp = (P * Y).sum().item(); fp = (P * (1 - Y)).sum().item(); fn = ((1 - P) * Y).sum().item()
    if (tp + fp) == 0 or (tp + fn) == 0: return 0.0, thr
    prec = tp / (tp + fp); rec = tp / (tp + fn)
    return (2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0), thr

def f1_zero_pad(pred_flat, target_flat, valid_flat, percentile, strict=False):
    """Convention B : NaN → 0, then quantile on pool with zeros."""
    p_z = torch.where(valid_flat, pred_flat, torch.zeros_like(pred_flat))
    t_z = torch.where(valid_flat, target_flat, torch.zeros_like(target_flat))
    thr = float(torch.quantile(t_z, percentile / 100.0))
    cmp = (lambda x: x > thr) if strict else (lambda x: x >= thr)
    P = cmp(p_z).float(); Y = cmp(t_z).float()
    tp = (P * Y).sum().item(); fp = (P * (1 - Y)).sum().item(); fn = ((1 - P) * Y).sum().item()
    if (tp + fp) == 0 or (tp + fn) == 0: return 0.0, thr
    prec = tp / (tp + fp); rec = tp / (tp + fn)
    return (2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0), thr

def f1_etccdi_per_pixel(pred_mm, target_mm, land_mask_np, clim_p99_np):
    """ETCCDI per-pixel threshold from train climatology.
    pred_mm, target_mm : torch [N, 1, H, W] in mm/day
    land_mask_np, clim_p99_np : numpy [H, W]
    Returns F1@p99 averaged per-pixel valid threshold + raw TP/FP/FN."""
    N, _, H, W = pred_mm.shape
    pred_np   = pred_mm.cpu().numpy().reshape(N, H, W)
    target_np = target_mm.cpu().numpy().reshape(N, H, W)
    valid_pixel = land_mask_np & np.isfinite(clim_p99_np)
    if not valid_pixel.any():
        return {'f1': float('nan'), 'tp': 0, 'fp': 0, 'fn': 0, 'n_valid_pixels': 0}
    thr = clim_p99_np[None, :, :]  # [1, H, W] broadcast
    pred_bin = pred_np >= thr
    target_bin = target_np >= thr
    valid_bcast = valid_pixel[None, :, :]
    tp = int((pred_bin & target_bin & valid_bcast).sum())
    fp = int((pred_bin & ~target_bin & valid_bcast).sum())
    fn = int((~pred_bin & target_bin & valid_bcast).sum())
    if (tp + fp) == 0 or (tp + fn) == 0:
        f1 = 0.0
    else:
        prec = tp / (tp + fp); rec = tp / (tp + fn)
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn, 'n_valid_pixels': int(valid_pixel.sum())}

def csi_at_threshold(pred_bin, target_bin):
    tp = int((pred_bin & target_bin).sum())
    fp = int((pred_bin & ~target_bin).sum())
    fn = int((~pred_bin & target_bin).sum())
    return tp / max(tp + fp + fn, 1)

def sedi_at_threshold(pred_bin, target_bin):
    """Symmetric Extremal Dependence Index (Ferro & Stephenson 2011)."""
    tp = int((pred_bin & target_bin).sum())
    fp = int((pred_bin & ~target_bin).sum())
    fn = int((~pred_bin & target_bin).sum())
    tn = int((~pred_bin & ~target_bin).sum())
    h = tp / max(tp + fn, 1); f = fp / max(fp + tn, 1)
    eps = 1e-12
    h = max(min(h, 1 - eps), eps); f = max(min(f, 1 - eps), eps)
    num = np.log(f) - np.log(h) - np.log(1 - f) + np.log(1 - h)
    den = np.log(f) + np.log(h) + np.log(1 - f) + np.log(1 - h)
    return float(num / den) if den != 0 else float('nan')

def fss_at_neighborhood(pred_bin, target_bin, n):
    """Roberts & Lean 2008 Fractions Skill Score. n = neighborhood size in pixels (odd)."""
    from scipy.ndimage import uniform_filter
    P = pred_bin.astype(np.float32); Q = target_bin.astype(np.float32)
    # Expert Recherche : mode='reflect' (Roberts & Lean 2008) avoids border zero-padding bias
    Pf = np.stack([uniform_filter(P[i], size=n, mode='reflect') for i in range(P.shape[0])])
    Qf = np.stack([uniform_filter(Q[i], size=n, mode='reflect') for i in range(Q.shape[0])])
    mse  = float(((Pf - Qf) ** 2).mean())
    norm = float((Pf ** 2 + Qf ** 2).mean())
    return 1.0 - mse / max(norm, 1e-12)

print('[Cell 7] helpers ready (4 F1 conventions, CSI, SEDI, FSS)')

In [ ]:
# === Cell 8 : ÉTAPE 2 — SAMPLING UNIQUE (N=16 batches × K=64) ===
# This is the one heavy step (~25 min). All subsequent measures are post-hoc.
import time

_t0 = time.time()

# Expert IA-B1 CRITICAL fix : the pipeline (config training_config.yaml:34
# nan_fill_strategy="mean") fills ocean NaN with spatial mean BEFORE the data
# reaches this notebook. The land_mask from Cell 3 lets us re-introduce the
# true land/sea distinction here, BEFORE caching. Without this, the F1 quantile
# is computed on synthetic ocean values and biases all conventions equally.
land_mask_tensor = torch.from_numpy(np.load(LAND_MASK_PATH)).to(DEVICE)   # [H, W] bool
print(f'[Cell 8] land_mask loaded : {int(land_mask_tensor.sum())} land pixels / {land_mask_tensor.numel()}')

cached_mu = []        # list of [1, 1, H, W]
cached_baseline = []
cached_target_resid = []   # residual = HR_log1p - baseline_log, OCEAN MASKED TO NAN
all_samples = []      # list of [K, 1, 1, H, W]

_count = 0
for sample in test_dataset:
    if _count >= N_BATCHES_FULL: break
    batch = convert_sample_to_batch(sample, builder, DEVICE)
    mu_total, baseline_log, target_residual = compute_mu_total_and_target(batch)

    # Apply land_mask : ocean pixels set to NaN so F1 / Pearson filter them naturally.
    # Broadcasting [1, 1, H, W] target with [H, W] mask :
    target_residual_masked = torch.where(land_mask_tensor.view(1, 1, *land_mask_tensor.shape),
                                          target_residual,
                                          torch.full_like(target_residual, float('nan')))

    # Expert IA-B3 fix : seed CUDA RNG as well, not just CPU
    torch.manual_seed(SEED + 1000 * _count)
    torch.cuda.manual_seed_all(SEED + 1000 * _count)
    samples_k = []
    for _ in range(K_SAMPLES_FULL):
        samples_k.append(_sample_once(diff_live, mu_total, baseline_log).detach().cpu())
    samples_k = torch.stack(samples_k, dim=0)   # [K, 1, 1, H, W]
    all_samples.append(samples_k)
    cached_mu.append(mu_total.detach().cpu())
    cached_baseline.append(baseline_log.detach().cpu())
    cached_target_resid.append(target_residual_masked.detach().cpu())
    _count += 1
    _elapsed = time.time() - _t0
    print(f'  batch {_count}/{N_BATCHES_FULL}  elapsed={_elapsed:.0f}s  '
          f'avg/batch={_elapsed/_count:.1f}s', flush=True)

# Cat into full tensors
all_samples       = torch.stack(all_samples, dim=0)            # [N, K, 1, 1, H, W]
cached_mu         = torch.cat(cached_mu, dim=0)                # [N, 1, H, W]
cached_baseline   = torch.cat(cached_baseline, dim=0)
cached_target_resid = torch.cat(cached_target_resid, dim=0)

print(f'\n[Étape 2] sampling done in {time.time()-_t0:.1f}s')
print(f'  all_samples       shape = {tuple(all_samples.shape)}  (N, K, 1, 1, H, W)')
print(f'  cached_mu         shape = {tuple(cached_mu.shape)}')
print(f'  cached_target_resid shape = {tuple(cached_target_resid.shape)}')

In [ ]:
# === Cell 9 : ÉTAPE 3a — Grille factorielle 2×2×2 (post-hoc, gratuit) ===
# (NaN_handling × N_BATCHES × K_SAMPLES) appliqué sur le SAMPLING UNIQUE de Cell 8.
# Mesure F1@p95 et F1@p99 sur 8 cellules + bootstrap CI 95%.

def _build_pred_full_residual(N, K):
    """Subset N batches and K samples from all_samples ; compute pred_full_residual = mu + delta_mean."""
    sub = all_samples[:N, :K]                         # [N, K, 1, 1, H, W]
    delta_mean = sub.mean(dim=1).squeeze(1)           # [N, 1, H, W]
    mu_sub = cached_mu[:N]                            # [N, 1, H, W]
    target_sub = cached_target_resid[:N]              # [N, 1, H, W]
    pred_full = mu_sub + delta_mean                   # residual = mu_HR + delta
    return pred_full, target_sub

def _bootstrap_f1(pred_full, target, valid_mask, f1_fn, percentile, n_boot=1000):
    """Bootstrap on the batch dimension."""
    pred_flat = pred_full.flatten()
    target_flat = target.flatten()
    valid_flat = valid_mask.flatten()
    N = pred_full.shape[0]
    sz_per_batch = pred_full.shape[-1] * pred_full.shape[-2] * pred_full.shape[-3]
    f1s = []
    rng = np.random.default_rng(SEED)
    for _ in range(n_boot):
        idx = rng.integers(0, N, size=N)
        sel = np.concatenate([np.arange(i*sz_per_batch, (i+1)*sz_per_batch) for i in idx])
        sel_t = torch.from_numpy(sel)
        f1, _ = f1_fn(pred_flat[sel_t], target_flat[sel_t], valid_flat[sel_t], percentile)
        if f1 == f1:    # not NaN
            f1s.append(f1)
    if not f1s: return float('nan'), float('nan'), float('nan')
    return float(np.median(f1s)), float(np.percentile(f1s, 2.5)), float(np.percentile(f1s, 97.5))

factorial_results = []
for N in (4, 16):
    for K in (32, 64):
        pred_full, target = _build_pred_full_residual(N, K)
        valid = torch.isfinite(target)
        valid_flat = valid.flatten()
        pred_flat  = pred_full.flatten()
        target_flat = target.flatten()
        n_valid_pix = int(valid_flat.sum())
        n_total_pix = int(valid_flat.numel())
        p_invalid = 1.0 - n_valid_pix / n_total_pix
        for conv_name, conv_fn in (('filter (strict >)', f1_filter), ('zero_pad (>=)', f1_zero_pad)):
            f1_99, thr_99 = conv_fn(pred_flat, target_flat, valid_flat, 99.0)
            f1_95, thr_95 = conv_fn(pred_flat, target_flat, valid_flat, 95.0)
            f1_99_med, f1_99_lo, f1_99_hi = _bootstrap_f1(pred_full, target, valid, conv_fn, 99.0, n_boot=1000)
            row = {
                'N_BATCHES': N, 'K_SAMPLES': K, 'convention': conv_name,
                'F1_p99': f1_99, 'F1_p95': f1_95,
                'threshold_p99': thr_99, 'threshold_p95': thr_95,
                'F1_p99_bootstrap_median': f1_99_med,
                'F1_p99_bootstrap_CI95_low': f1_99_lo,
                'F1_p99_bootstrap_CI95_high': f1_99_hi,
                'n_valid_pixels': n_valid_pix,
                'n_total_pixels': n_total_pix,
                'p_invalid': p_invalid,
            }
            factorial_results.append(row)

print('=' * 100)
print('FACTORIAL GRID 2×2×2 -- ALL ON THE SAME SAMPLING RUN (N=16, K=64 SAMPLED ONCE)')
print('=' * 100)
print(f'{"N":>3} {"K":>3} {"convention":<22} {"F1@p99":>8} {"F1@p95":>8} {"thr@p99":>10} '
      f'{"bootstrap median":>17} {"CI95":>22}')
print('-' * 100)
for r in factorial_results:
    ci = f'[{r["F1_p99_bootstrap_CI95_low"]:.3f},{r["F1_p99_bootstrap_CI95_high"]:.3f}]'
    print(f'{r["N_BATCHES"]:>3} {r["K_SAMPLES"]:>3} {r["convention"]:<22} '
          f'{r["F1_p99"]:>8.4f} {r["F1_p95"]:>8.4f} {r["threshold_p99"]:>10.4f} '
          f'{r["F1_p99_bootstrap_median"]:>17.4f} {ci:>22}')
print()
print(f'(p_invalid in tensors = {factorial_results[0]["p_invalid"]:.4f}  '
      f'-> {int(100*factorial_results[0]["p_invalid"])}% of pixels are NaN in target_residual)')

In [ ]:
# === Cell 10 : ÉTAPE 3b — F1 ETCCDI per-pixel + CSI + SEDI + FSS sur Phase 6 dualpath ===
# Reconstruction full HR in mm/day, then per-pixel climatology threshold.
import numpy as np

_clim = np.load(CLIM_PATH)
clim_p99_np = _clim['clim_p99']    # mm/day
clim_p95_np = _clim['clim_p95']
land_mask_np = _clim['land_mask']
print(f'[Étape 3b] climatology loaded : p99 range [{np.nanmin(clim_p99_np):.2f}, '
      f'{np.nanmax(clim_p99_np):.2f}] mm/day  n_land = {int(land_mask_np.sum())}')

# Reconstruct HR in log1p then mm/day (using N=16, K=64 = full)
pred_resid, target_resid = _build_pred_full_residual(N_BATCHES_FULL, K_SAMPLES_FULL)
baseline_full = cached_baseline[:N_BATCHES_FULL]
pred_log1p   = baseline_full + pred_resid       # full HR in log1p (mm/day after expm1)
target_log1p = baseline_full + target_resid
# Expert Climat R2 fix : pipeline applies log1p(x + PRECIPITATION_DELTA),
# so inverse is expm1(y) - PRECIPITATION_DELTA. Otherwise +0.01 mm/day bias.
# Expert Climat R3 fix : clamp max=500 mm/day (world record 24h ~ Cilaos 1966).
pred_mm   = (torch.expm1(pred_log1p)   - PRECIPITATION_DELTA).clamp(min=0, max=500)
target_mm = (torch.expm1(target_log1p) - PRECIPITATION_DELTA).clamp(min=0, max=500)

etccdi = f1_etccdi_per_pixel(pred_mm, target_mm, land_mask_np, clim_p99_np)
etccdi_p95 = f1_etccdi_per_pixel(pred_mm, target_mm, land_mask_np, clim_p95_np)
print()
print('=' * 78)
print('ÉTAPE 3b : F1 ETCCDI per-pixel on PHASE 6 DUAL-PATH (live weights)')
print('=' * 78)
print(f'  F1 @ ETCCDI p95 per-pixel   = {etccdi_p95["f1"]:.4f}  '
      f'(TP={etccdi_p95["tp"]} FP={etccdi_p95["fp"]} FN={etccdi_p95["fn"]})')
print(f'  F1 @ ETCCDI p99 per-pixel   = {etccdi["f1"]:.4f}  '
      f'(TP={etccdi["tp"]} FP={etccdi["fp"]} FN={etccdi["fn"]})')
print(f'  n_land pixels with clim     = {etccdi["n_valid_pixels"]}')

# CSI / SEDI / FSS at the same per-pixel threshold
pred_np = pred_mm.cpu().numpy().reshape(N_BATCHES_FULL, H_HR, W_HR)
target_np = target_mm.cpu().numpy().reshape(N_BATCHES_FULL, H_HR, W_HR)
valid_pixel = land_mask_np & np.isfinite(clim_p99_np)
pred_bin = (pred_np >= clim_p99_np[None]) & valid_pixel[None]
target_bin = (target_np >= clim_p99_np[None]) & valid_pixel[None]
csi = csi_at_threshold(pred_bin, target_bin)
sedi = sedi_at_threshold(pred_bin, target_bin)
fss_9 = fss_at_neighborhood(pred_bin, target_bin, 9)
fss_25 = fss_at_neighborhood(pred_bin, target_bin, 25)
print(f'  CSI @ ETCCDI p99            = {csi:.4f}')
print(f'  SEDI @ ETCCDI p99           = {sedi:.4f}')
print(f'  FSS @ ETCCDI p99 (n=9 px)   = {fss_9:.4f}')
print(f'  FSS @ ETCCDI p99 (n=25 px)  = {fss_25:.4f}')

In [ ]:
# === Cell 11 : ÉTAPE 4 — Reload existing JSONs + synthesis ===
import json

REF_PATHS = {
    'noncausal_v4':   DRIVE_ROOT / 'ckpt_noncausal' / 'final_validation_metrics.json',
    'V5_causal':      DRIVE_ROOT / 'ckpt_v2_corrdiff_normal' / 'final_validation_metrics.json',
    'seed_42_v1':     DRIVE_ROOT / 'oracle_9node' / 'seed_42' / 'final_validation_metrics.json',
    'seed_42_v2':     DRIVE_ROOT / 'oracle_full'  / 'seed_42_v2' / 'final_validation_metrics.json',
    'phase6_dualpath':DRIVE_ROOT / 'ckpt_phase6_dualpath' / 'final_validation_metrics.json',
}
existing = {}
for name, p in REF_PATHS.items():
    if p.exists():
        try:
            existing[name] = json.loads(p.read_text(encoding='utf-8'))
        except Exception as _e:
            print(f'[warn] could not load {name} : {_e}')

print('=' * 100)
print('SYNTHÈSE — Phase 6 dual-path vs reference JSONs')
print('=' * 100)
print()
print('=' * 100)
print('SECTION A : ETCCDI per-pixel (publication-ready) -- Phase 6 dualpath ONLY')
print('=' * 100)
print(f'{"Model":<25}{"F1@p99":>10}{"F1@p95":>10}{"CSI@p99":>10}{"SEDI@p99":>10}{"FSS@n=9":>10}{"FSS@n=25":>10}')
print('-' * 100)
print(f'{"phase6 dualpath (LIVE)":<25}{etccdi["f1"]:>10.4f}{etccdi_p95["f1"]:>10.4f}'
      f'{csi:>10.4f}{sedi:>10.4f}{fss_9:>10.4f}{fss_25:>10.4f}')
print()
print('=' * 100)
print('SECTION B : Legacy Conv B (zero+pooled) -- NOT COMPARABLE WITH SECTION A')
print('   Existing JSONs from previous runs. Must be re-evaluated in ETCCDI mode')
print('   before any cross-model conclusion can be drawn.')
print('=' * 100)
print(f'{"Model":<25}{"F1@p99":>10}{"F1@p95":>10}{"Pearson":>10}{"RMSE":>10}  *INCOMPARABLE*')
print('-' * 100)
for name, j in existing.items():
    f99 = j.get('f1_extremes', {}).get('p99', float('nan'))
    f95 = j.get('f1_extremes', {}).get('p95', float('nan'))
    pg  = j.get('pearson_corr', {}).get('global', float('nan'))
    rmse = j.get('rmse', float('nan'))
    print(f'{name:<25}{f99:>10.4f}{f95:>10.4f}{pg:>10.4f}{rmse:>10.4f}  [Conv B]')

# Best convention from factorial : N=16, K=64, choose 'zero_pad (>=)' to match phase6 protocol
_best_match = next(r for r in factorial_results
                   if r['N_BATCHES']==N_BATCHES_FULL and r['K_SAMPLES']==K_SAMPLES_FULL
                   and 'zero_pad' in r['convention'])
_best_filter = next(r for r in factorial_results
                    if r['N_BATCHES']==N_BATCHES_FULL and r['K_SAMPLES']==K_SAMPLES_FULL
                    and 'filter' in r['convention'])
print(f'\n{"phase6 (live, N=16 K=64)":<20}{_best_match["F1_p99"]:>10.4f}'
      f'{_best_match["F1_p95"]:>10.4f}{"--":>10}{"--":>10}{"B (zero+pooled)":>30}')
print(f'{"phase6 (live, N=16 K=64)":<20}{_best_filter["F1_p99"]:>10.4f}'
      f'{_best_filter["F1_p95"]:>10.4f}{"--":>10}{"--":>10}{"A (filter, strict >)":>30}')
print(f'{"phase6 ETCCDI per-pix":<20}{etccdi["f1"]:>10.4f}{etccdi_p95["f1"]:>10.4f}'
      f'{"--":>10}{"--":>10}{"ETCCDI per-pixel":>30}')

# Diagnostic narrative
print()
print('=' * 100)
print('DIAGNOSTIC')
print('=' * 100)
_factor_delta = abs(_best_match['F1_p99'] - _best_filter['F1_p99'])
if _factor_delta > 0.10:
    print(f'• Convention (NaN handling) changes F1@p99 by {_factor_delta:.3f} -> NON-NEGLIGIBLE.')
else:
    print(f'• Convention changes F1@p99 by only {_factor_delta:.3f} -> NaN handling NOT the main factor.')

_p_inv = factorial_results[0]['p_invalid']
if _p_inv < 0.01:
    print(f'• p_invalid in tensors = {_p_inv:.4f} (essentially 0) -> as expert Climat predicted,')
    print('  the pipeline `nan_fill_strategy=mean` removed NaN BEFORE eval. Convention A and B operate')
    print('  on identical data, the discrepancy 0.539 vs 0.175 must come from elsewhere (N, K, sampling).')
else:
    print(f'• p_invalid in tensors = {_p_inv:.4f} -> NaN are present and Convention matters.')

# Save unified JSON
payload = {
    'protocol': 'unified_etccdi_v1',
    'sampling': {'N_BATCHES': N_BATCHES_FULL, 'K_SAMPLES': K_SAMPLES_FULL,
                  'N_STEPS_DIFF': N_STEPS_DIFF, 'scheduler': 'edm_karras', 'cfg_scale': 0.0,
                  'seed': SEED, 'ckpt': str(CKPT_STAGE2), 'weights': 'live'},
    'factorial_grid_2x2x2': factorial_results,
    'etccdi_per_pixel': {
        'p95': etccdi_p95, 'p99': etccdi,
        'csi_p99': csi, 'sedi_p99': sedi,
        'fss_p99_n9': fss_9, 'fss_p99_n25': fss_25,
        'land_mask_n_pixels': int(land_mask_np.sum()),
        'climatology_source': str(CLIM_PATH),
    },
    'reference_existing_jsons': {name: {
        'F1_p99': j.get('f1_extremes', {}).get('p99'),
        'F1_p95': j.get('f1_extremes', {}).get('p95'),
        'Pearson_global': j.get('pearson_corr', {}).get('global'),
        'RMSE': j.get('rmse'),
        'convention': 'B (zero-pad + pooled, NOT ETCCDI)',
    } for name, j in existing.items()},
    'caveats': [
        'Existing JSONs use Convention B (zero-pad). For publication-grade comparison, ALL models must be re-evaluated in ETCCDI per-pixel mode.',
        'p_invalid in tensors is ~0 because pipeline.nan_fill_strategy="mean" filled NaN before eval. The land_mask + ETCCDI threshold is built independently from raw NetCDF.',
        'Bootstrap CI uses 200 resamples on the batch axis.',
        'ETCCDI F1 in mm/day requires expm1(baseline+mu_HR+delta). Numerical stability OK if log1p inputs were in [0, 10] range.',
    ],
}
OUT_JSON.write_text(json.dumps(payload, indent=2, default=str), encoding='utf-8')
print(f'\nSaved : {OUT_JSON}')

try:
    from google.colab import files
    files.download(str(OUT_JSON))
except Exception:
    pass